# System A: GRU language model (Phase 3)

Trains the GRU generator in two stages and saves the checkpoints named in the plan:

1. **Stage 1** on the 18,850 modern AABB quatrains (no traits, intensity `none`) → `gen_A_stage1_best.pt`
2. **Stage 2** on the 3,382 training disses, starting from Stage 1, at 1/5 of the learning rate → `gen_A_stage2_best.pt`

Each sequence is `<traits> … <intensity> … <rhymes> w1 w2 w3 w4 <poem> line1 <nl> … line4 <eos>`, and the loss
covers the poem only (see `generation/sequences.py`; the model and training loop are in `generation/gru_lm.py`).

**On Colab:** Runtime → Change runtime type → **T4 GPU**, then Run all. The repo is cloned from GitHub and the
checkpoints are saved to Google Drive (`MyDrive/CS425/checkpoints`), so they survive a disconnect. If the runtime
disconnects, run all again: training resumes from the last finished epoch, and a finished stage is not retrained.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/SchoolWork-awork2103/CS425-Project.git"
BRANCH = "feat/phase0"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/CS425/checkpoints"
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # a few batches only, to check the notebook runs end to end

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    root = "/content/CS425-Project"
    if os.path.exists(root):
        subprocess.run(["git", "-C", root, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, root], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "pyyaml"], check=True)
    os.chdir(root)
    CKPT_DIR = DRIVE_CKPT_DIR
else:
    if os.path.basename(os.getcwd()) == "generation":
        os.chdir("..")
    CKPT_DIR = "checkpoints"
if SMOKE_TEST:
    CKPT_DIR = os.path.join(CKPT_DIR, "smoke_test")
sys.path[:0] = [os.getcwd(), os.path.join(os.getcwd(), "generation")]
print("project:", os.getcwd(), "| checkpoints:", CKPT_DIR, "| smoke test:", SMOKE_TEST)

In [ ]:
import json, random
import torch
from utils import load_config, project_path, set_seed
from sequences import PoemCodec, load_split, read_jsonl
from gru_lm import GRULM, train_stage, run_epoch, load_model, sample_report
from sequences import SequenceDataset, make_collate

cfg = load_config()
SEED = set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
codec = PoemCodec(cfg)

stage1 = {s: load_split(codec, 1, s, cfg) for s in ("train", "val")}
stage2 = {s: load_split(codec, 2, s, cfg) for s in ("train", "val", "test")}
print("device:", device, "| vocab:", codec.vocab_size)
print({f"stage1 {k}": len(v) for k, v in stage1.items()}, {f"stage2 {k}": len(v) for k, v in stage2.items()})

In [ ]:
# Hyperparameters (pipeline.md, System A)
MODEL = dict(emb_dim=300, hidden=512, layers=2, dropout=0.3)
BATCH_SIZE = 64
LR_STAGE1 = 2e-3
LR_STAGE2 = LR_STAGE1 / 5          # Stage 2 fine-tunes at about 1/5 of the Stage 1 learning rate
MAX_EPOCHS_STAGE1, MAX_EPOCHS_STAGE2 = 40, 30
PATIENCE = 3                       # early stopping: epochs without a better validation loss
SAMPLING = dict(temperature=0.8, top_p=0.9, rep_penalty=1.2)

MAX_BATCHES = 3 if SMOKE_TEST else None
if SMOKE_TEST:
    MAX_EPOCHS_STAGE1 = MAX_EPOCHS_STAGE2 = 2

## Stage 1: pretraining on the quatrain corpus
The model learns English, the four-line AABB shape and ending lines on the requested rhyme words.

In [ ]:
model = GRULM(codec.vocab_size, pad_id=codec.pad, **MODEL).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
hist1 = train_stage(model, stage1["train"], stage1["val"], codec.pad, device, CKPT_DIR, "gen_A_stage1",
                    lr=LR_STAGE1, max_epochs=MAX_EPOCHS_STAGE1, patience=PATIENCE, batch_size=BATCH_SIZE,
                    seed=SEED, max_batches=MAX_BATCHES)

In [ ]:
# Stage 1 samples on validation prompts (unseen topics): rhyme words from real validation quatrains
rows = read_jsonl(project_path(cfg["paths"]["corpus_dir"]) / "quatrains_val.jsonl")
random.Random(SEED).shuffle(rows)
prompts1 = [{"traits": [], "intensity": "none", "end_words": [w for p in r["rhymes"] for w in p]}
            for r in rows[:5 if SMOKE_TEST else 50]]
report1 = sample_report(model, codec, prompts1, device, n=4, show=5, **SAMPLING)

## Stage 2: fine-tuning on the disses
Starts from the best Stage 1 weights and learns to fill in the traits and intensity and make it a roast.

In [ ]:
model, _ = load_model(os.path.join(CKPT_DIR, "gen_A_stage1_best.pt"), device)
hist2 = train_stage(model, stage2["train"], stage2["val"], codec.pad, device, CKPT_DIR, "gen_A_stage2",
                    lr=LR_STAGE2, max_epochs=MAX_EPOCHS_STAGE2, patience=PATIENCE, batch_size=BATCH_SIZE,
                    seed=SEED, max_batches=MAX_BATCHES)

In [ ]:
# Stage 2 samples: validation disses (training traits) and held-out traits from the test set
diss_dir = project_path(cfg["paths"]["diss_dir"])
val_rows, test_rows = read_jsonl(diss_dir / "val.jsonl"), read_jsonl(diss_dir / "test.jsonl")
random.Random(SEED).shuffle(val_rows)
k = 5 if SMOKE_TEST else 40
to_prompt = lambda r: {"traits": r["traits"], "intensity": r["intensity"], "end_words": r["end_words"]}
heldout_rows = [r for r in test_rows if r["kind"] == "heldout"]
random.Random(SEED).shuffle(heldout_rows)

print("=== training traits (validation disses)")
report2 = sample_report(model, codec, [to_prompt(r) for r in val_rows[:k]], device, n=4, show=5, **SAMPLING)
print("\n=== held-out traits (never seen in training)")
report2_heldout = sample_report(model, codec, [to_prompt(r) for r in heldout_rows[:k]], device, n=4, show=5,
                                **SAMPLING)

In [ ]:
# Test loss on the diss test set, separately for training traits and held-out traits
collate = make_collate(codec.pad)
def test_loss(rows):
    ex = [codec.example(r["lines"], r["end_words"], r["traits"], r["intensity"]) for r in rows]
    dl = torch.utils.data.DataLoader(SequenceDataset(ex), BATCH_SIZE, collate_fn=collate)
    return run_epoch(model, dl, device, max_batches=MAX_BATCHES)

import math
seen_rows = [r for r in test_rows if r["kind"] != "heldout"]
test = {"seen_traits": test_loss(seen_rows), "heldout_traits": test_loss(heldout_rows)}
for name, loss in test.items():
    print(f"test {name:15s} loss {loss:.4f}  perplexity {math.exp(loss):6.2f}")

summary = {"model": MODEL, "batch_size": BATCH_SIZE, "lr": [LR_STAGE1, LR_STAGE2], "sampling": SAMPLING,
           "stage1": {"epochs": len(hist1), "best_val_loss": min(h["val_loss"] for h in hist1), "samples": report1},
           "stage2": {"epochs": len(hist2), "best_val_loss": min(h["val_loss"] for h in hist2),
                      "samples": report2, "samples_heldout": report2_heldout},
           "test_loss": test}
json.dump(summary, open(os.path.join(CKPT_DIR, "gen_A_summary.json"), "w"), indent=2)
print("saved", os.path.join(CKPT_DIR, "gen_A_summary.json"))